In [ ]:
"""
shap_analysis.py
-----------------
SHAP feature importance analysis for RQ4.

For each target variable, this script:
    1. Trains the best-performing multimodal model on the full dataset
       to extract global SHAP feature importance
    2. Computes SHAP values per outer fold to assess stability
    3. Measures Jaccard similarity of top-10 features across folds
    4. Produces beeswarm and stability plots

Outputs (saved to RESULTS_DIR/shap/):
    shap_importance_<target>.csv   — mean |SHAP| per feature
    shap_jaccard_<target>.csv      — pairwise Jaccard similarity across folds
    shap_beeswarm_<target>.png     — beeswarm plot
    shap_stability_<target>.png    — top-10 feature stability heatmap
    shap_summary.csv               — mean Jaccard per target
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap

from itertools import combinations
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from xgboost                 import XGBRegressor

warnings.filterwarnings("ignore")

# =============================================================================
# CELL 1 — Paths (update these)
# =============================================================================

OUT_DIR      = r"C:\Users\bwldi\OneDrive\Thesis\Right DATA"        # <-- change this
FEATURES_CSV = os.path.join(OUT_DIR, r"C:\Users\bwldi\OneDrive\Thesis\Right DATA\full_features.csv")
SHAP_DIR     = os.path.join(OUT_DIR, "results", "shap")
os.makedirs(SHAP_DIR, exist_ok=True)


# =============================================================================
# CELL 2 — Load data and define feature sets
# =============================================================================

df = pd.read_csv(FEATURES_CSV)
print(f"Loaded: {df.shape}")

STRUCTURAL_COLS = [
    "speaking_time_mean", "speaking_time_std", "gini_speaking_time",
    "max_speaker_share", "num_turns", "turns_per_minute",
    "mean_turn_duration", "std_turn_duration", "overlap_ratio",
    "num_interruptions_proxy", "silence_ratio", "avg_pause_duration",
]
DA_COLS        = [c for c in df.columns if c.startswith("da_prop_")]
SENTIMENT_COLS = [
    "vader_pos_mean", "vader_pos_std",
    "vader_neg_mean", "vader_neg_std",
    "vader_neu_mean", "vader_neu_std",
    "vader_compound_mean", "vader_compound_std",
]
MULTIMODAL_COLS = STRUCTURAL_COLS + DA_COLS + SENTIMENT_COLS

TARGET_COLS = [
    "overall_performance",
    "satisfaction",
    "cohesiveness",
    "leadership",
    "information_processing",
]

# Best model per target from modelling results
# (multimodal ENR was best for most, but SHAP is tree-based so we use XGB
#  for the multimodal condition — this is consistent with how XGBoost SHAP
#  values are exact rather than approximate)
# We use XGBoost throughout for SHAP since TreeExplainer gives exact values.
BEST_XGB_PARAMS = {
    "overall_performance":    {"n_estimators": 200, "max_depth": 3,  "learning_rate": 0.1,  "subsample": 0.8},
    "satisfaction":           {"n_estimators": 200, "max_depth": 3,  "learning_rate": 0.1,  "subsample": 0.8},
    "cohesiveness":           {"n_estimators": 200, "max_depth": 3,  "learning_rate": 0.1,  "subsample": 0.8},
    "leadership":             {"n_estimators": 200, "max_depth": 3,  "learning_rate": 0.1,  "subsample": 0.8},
    "information_processing": {"n_estimators": 200, "max_depth": 3,  "learning_rate": 0.1,  "subsample": 0.8},
}

RANDOM_STATE = 42
N_OUTER      = 10


# =============================================================================
# CELL 3 — Helper: Jaccard similarity between two sets
# =============================================================================

def jaccard(set_a, set_b):
    a, b = set(set_a), set(set_b)
    if len(a | b) == 0:
        return 0.0
    return len(a & b) / len(a | b)


# =============================================================================
# CELL 4 — SHAP analysis per target
# =============================================================================

summary_rows = []

for target in TARGET_COLS:
    print(f"\n{'='*60}")
    print(f"Target: {target}")
    print(f"{'='*60}")

    y_raw    = df[target].values
    X_raw    = df[MULTIMODAL_COLS].values
    y_binned = pd.qcut(y_raw, q=3, labels=False, duplicates="drop")

    outer_cv = StratifiedKFold(
        n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE
    )

    params = BEST_XGB_PARAMS[target]

    # ------------------------------------------------------------------
    # Part A: Per-fold SHAP values for stability analysis
    # ------------------------------------------------------------------
    fold_top10 = []   # top-10 feature names per fold
    fold_shap  = []   # mean |SHAP| vector per fold

    for fold_idx, (train_idx, test_idx) in enumerate(
            outer_cv.split(X_raw, y_binned)):

        X_train = X_raw[train_idx]
        y_train = y_raw[train_idx]
        X_test  = X_raw[test_idx]

        # Scale
        scaler  = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s  = scaler.transform(X_test)

        # Fit XGB with best params
        model = XGBRegressor(
            **params,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
        )
        model.fit(X_train_s, y_train)

        # SHAP values on test fold
        explainer  = shap.TreeExplainer(model)
        shap_vals  = explainer.shap_values(X_test_s)  # shape (n_test, n_features)

        mean_abs_shap = np.abs(shap_vals).mean(axis=0)
        fold_shap.append(mean_abs_shap)

        # Top-10 features for this fold
        top10_idx = np.argsort(mean_abs_shap)[::-1][:10]
        top10_names = [MULTIMODAL_COLS[i] for i in top10_idx]
        fold_top10.append(top10_names)
        print(f"  Fold {fold_idx+1:2d} top-3: {top10_names[:3]}")

    # ------------------------------------------------------------------
    # Part B: Jaccard stability across folds
    # ------------------------------------------------------------------
    jaccard_scores = []
    for i, j in combinations(range(N_OUTER), 2):
        jaccard_scores.append(jaccard(fold_top10[i], fold_top10[j]))

    mean_jaccard = np.mean(jaccard_scores)
    std_jaccard  = np.std(jaccard_scores)
    print(f"\n  Mean Jaccard similarity (top-10): {mean_jaccard:.3f} ± {std_jaccard:.3f}")

    # Save Jaccard scores
    jaccard_df = pd.DataFrame({
        "fold_i": [i for i, j in combinations(range(N_OUTER), 2)],
        "fold_j": [j for i, j in combinations(range(N_OUTER), 2)],
        "jaccard": jaccard_scores,
    })
    jaccard_df.to_csv(
        os.path.join(SHAP_DIR, f"shap_jaccard_{target}.csv"), index=False
    )

    # ------------------------------------------------------------------
    # Part C: Global SHAP on full dataset for beeswarm plot
    # ------------------------------------------------------------------
    scaler_full = StandardScaler()
    X_scaled    = scaler_full.fit_transform(X_raw)

    model_full = XGBRegressor(
        **params,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )
    model_full.fit(X_scaled, y_raw)

    explainer_full = shap.TreeExplainer(model_full)
    shap_full      = explainer_full.shap_values(X_scaled)

    # Feature importance dataframe
    mean_abs_full = np.abs(shap_full).mean(axis=0)
    importance_df = pd.DataFrame({
        "feature":        MULTIMODAL_COLS,
        "mean_abs_shap":  mean_abs_full,
        "modality":       (
            ["structural"] * len(STRUCTURAL_COLS) +
            ["da"]         * len(DA_COLS) +
            ["sentiment"]  * len(SENTIMENT_COLS)
        ),
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    importance_df.to_csv(
        os.path.join(SHAP_DIR, f"shap_importance_{target}.csv"), index=False
    )

    print(f"\n  Top-10 global features:")
    print(importance_df.head(10)[["feature", "modality", "mean_abs_shap"]].to_string(index=False))

    # ------------------------------------------------------------------
    # Part D: Beeswarm plot (top 15 features)
    # ------------------------------------------------------------------
    top15        = importance_df.head(15)["feature"].tolist()
    top15_idx    = [MULTIMODAL_COLS.index(f) for f in top15]
    shap_top15   = shap_full[:, top15_idx]
    X_top15      = X_scaled[:, top15_idx]

    # Readable feature labels
    label_map = {
        "speaking_time_mean":      "Speaking time (mean)",
        "speaking_time_std":       "Speaking time (SD)",
        "gini_speaking_time":      "Gini coefficient",
        "max_speaker_share":       "Max speaker share",
        "num_turns":               "Number of turns",
        "turns_per_minute":        "Turns per minute",
        "mean_turn_duration":      "Turn duration (mean)",
        "std_turn_duration":       "Turn duration (SD)",
        "overlap_ratio":           "Overlap ratio",
        "num_interruptions_proxy": "Interruptions (proxy)",
        "silence_ratio":           "Silence ratio",
        "avg_pause_duration":      "Avg pause duration",
        "vader_pos_mean":          "VADER positive (mean)",
        "vader_pos_std":           "VADER positive (SD)",
        "vader_neg_mean":          "VADER negative (mean)",
        "vader_neg_std":           "VADER negative (SD)",
        "vader_neu_mean":          "VADER neutral (mean)",
        "vader_neu_std":           "VADER neutral (SD)",
        "vader_compound_mean":     "VADER compound (mean)",
        "vader_compound_std":      "VADER compound (SD)",
    }
    # DA features: strip da_prop_ prefix and replace underscores
    for col in DA_COLS:
        label_map[col] = col.replace("da_prop_", "DA: ").replace("_", " ")

    labels = [label_map.get(f, f) for f in top15]

    # Colour by modality
    modality_colours = {
        "structural": "#2C6FAC",
        "da":         "#E07B3F",
        "sentiment":  "#4DAF4A",
    }
    modalities = importance_df.set_index("feature")["modality"].to_dict()

    fig, ax = plt.subplots(figsize=(9, 6))
    shap.summary_plot(
        shap_top15,
        X_top15,
        feature_names=labels,
        show=False,
        plot_type="dot",
        color_bar=True,
        max_display=15,
        plot_size=None,
    )
    ax = plt.gca()
    ax.set_title(
        f"SHAP feature importance — {target.replace('_', ' ')}",
        fontsize=12, pad=10
    )

    # Add modality legend
    legend_patches = [
        mpatches.Patch(color=c, label=m.capitalize())
        for m, c in modality_colours.items()
    ]
    ax.legend(handles=legend_patches, loc="lower right",
              fontsize=9, title="Modality", title_fontsize=9)

    plt.tight_layout()
    plt.savefig(
        os.path.join(SHAP_DIR, f"shap_beeswarm_{target}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()
    print(f"  Beeswarm saved.")

    # ------------------------------------------------------------------
    # Part E: Stability heatmap — which features appear in top-10 per fold
    # ------------------------------------------------------------------
    all_top_features = sorted(set(f for fold in fold_top10 for f in fold))
    stability_matrix = pd.DataFrame(
        {f"Fold {i+1}": [1 if f in fold_top10[i] else 0
                         for f in all_top_features]
         for i in range(N_OUTER)},
        index=all_top_features
    )
    # Sort by total appearances
    stability_matrix["total"] = stability_matrix.sum(axis=1)
    stability_matrix = stability_matrix.sort_values("total", ascending=False).drop(columns="total")

    fig, ax = plt.subplots(figsize=(10, max(4, len(stability_matrix) * 0.4)))
    im = ax.imshow(stability_matrix.values, aspect="auto",
                   cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_OUTER))
    ax.set_xticklabels([f"F{i+1}" for i in range(N_OUTER)], fontsize=9)
    ax.set_yticks(range(len(stability_matrix)))
    ax.set_yticklabels(
        [label_map.get(f, f) for f in stability_matrix.index],
        fontsize=9
    )
    ax.set_title(
        f"Top-10 feature stability across folds — {target.replace('_', ' ')}\n"
        f"Mean Jaccard = {mean_jaccard:.3f} ± {std_jaccard:.3f}",
        fontsize=11
    )
    plt.colorbar(im, ax=ax, shrink=0.5, label="In top-10")
    plt.tight_layout()
    plt.savefig(
        os.path.join(SHAP_DIR, f"shap_stability_{target}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()
    print(f"  Stability heatmap saved.")

    summary_rows.append({
        "target":        target,
        "mean_jaccard":  round(mean_jaccard, 4),
        "std_jaccard":   round(std_jaccard,  4),
    })

# =============================================================================
# CELL 5 — Save summary
# =============================================================================

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(SHAP_DIR, "shap_summary.csv"), index=False)

print("\n\n=== SHAP STABILITY SUMMARY ===")
print(summary_df.to_string(index=False))
print(f"\nAll outputs saved to: {SHAP_DIR}")